# Demo 6 of 6 - Cost by workload and by client

**Why we are doing this.** Kenny's complaint: they cannot track cost by workload or by health-plan client.

**What we will prove.**
1. Because each workload has its own warehouse, cost by workload is a simple query.
2. Every query is tagged with the client and workload, so cost can be attributed to each client even on a shared warehouse.
3. Serverless features (Snowpipe, dynamic tables, quality checks) are metered separately and visible too.
4. A budget with email alerts acts as a guardrail.

**Important timing note.** These views in SNOWFLAKE.ACCOUNT_USAGE are delayed: warehouse metering by up to about 3 hours, tag references by about 2 hours, and per-query attribution by up to about 8 hours. Run this notebook the morning of the call to confirm the numbers are populated from your rehearsal.

**How to read this notebook.** Every cell has one of four labels:
- **SETUP - ALREADY DONE (read only):** code that already exists in the account. It is shown so the engineers can see how it was built. You do not run it.
- **RUN DURING DEMO:** a SQL cell you run live. Click inside it and press the run button (or Cmd+Enter).
- **UI STEP:** something you do in the Snowsight screens or the AWS console, not in the notebook.
- **SAY THIS:** your talk track for that moment.

### SETUP - ALREADY DONE (read only, do not run)
**Tagging**

```sql
-- Each warehouse carries a WORKLOAD tag
CREATE TAG SOMATUS_DEMO.OPS.WORKLOAD;
ALTER WAREHOUSE WH_INGEST    SET TAG SOMATUS_DEMO.OPS.WORKLOAD = 'ingest';
ALTER WAREHOUSE WH_TRANSFORM SET TAG SOMATUS_DEMO.OPS.WORKLOAD = 'transform';
ALTER WAREHOUSE WH_BI        SET TAG SOMATUS_DEMO.OPS.WORKLOAD = 'bi';
-- Each client job sets a QUERY_TAG, for example on the task T_CLIENT_HUMANA:
--   QUERY_TAG = 'client=HUMANA_DEMO;workload=transform'
```

### SETUP - ALREADY DONE (read only, do not run)
**Budget**

```sql
-- OPS.SOMATUS_DE_BUDGET: a 200-credit monthly limit on the demo warehouses, with email notification.
CALL SOMATUS_DEMO.OPS.SOMATUS_DE_BUDGET!SET_SPENDING_LIMIT(200);
```

### RUN DURING DEMO
Set the role, warehouse and database. Run this first, every time you open the notebook.

In [ ]:
%%sql -r ctx
USE ROLE ACCOUNTADMIN;
USE DATABASE SOMATUS_DEMO;
USE WAREHOUSE WH_INGEST;

## Step 1 - How things are tagged

### RUN DURING DEMO
Show the workload tag on the transform warehouse.

In [ ]:
%%sql -r tag
SELECT SYSTEM$GET_TAG('SOMATUS_DEMO.OPS.WORKLOAD', 'WH_TRANSFORM', 'WAREHOUSE') AS WORKLOAD_TAG;

### RUN DURING DEMO
Show the per-client jobs. Scroll right to the **query_tag** column: each one names its client.

In [ ]:
%%sql -r client_tasks
SHOW TASKS LIKE 'T_CLIENT_%' IN SCHEMA SOMATUS_DEMO.OPS;

## Step 2 - Cost by workload

### RUN DURING DEMO
Credits used per workload over the last 7 days.

In [ ]:
%%sql -r by_workload
SELECT COALESCE(t.TAG_VALUE, 'untagged') AS WORKLOAD, m.WAREHOUSE_NAME,
       ROUND(SUM(m.CREDITS_USED), 3) AS CREDITS
FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY m
LEFT JOIN SNOWFLAKE.ACCOUNT_USAGE.TAG_REFERENCES t
  ON t.DOMAIN = 'WAREHOUSE' AND t.OBJECT_NAME = m.WAREHOUSE_NAME
 AND t.TAG_NAME = 'WORKLOAD' AND t.TAG_DATABASE = 'SOMATUS_DEMO'
WHERE m.START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
  AND m.WAREHOUSE_NAME LIKE 'WH\\_%' ESCAPE '\\'
GROUP BY 1, 2 ORDER BY 3 DESC;

## Step 3 - Cost by client

### RUN DURING DEMO
Credits attributed to each client and workload, calculated query by query. This works even when clients share a warehouse.

In [ ]:
%%sql -r by_client
SELECT REGEXP_SUBSTR(QUERY_TAG, 'client=([A-Z_]+)', 1, 1, 'e')  AS CLIENT,
       REGEXP_SUBSTR(QUERY_TAG, 'workload=([a-z]+)', 1, 1, 'e') AS WORKLOAD,
       COUNT(*) AS QUERIES,
       ROUND(SUM(CREDITS_ATTRIBUTED_COMPUTE), 4) AS CREDITS
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY
WHERE START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP()) AND QUERY_TAG ILIKE '%client=%'
GROUP BY 1, 2 ORDER BY CREDITS DESC;

### RUN DURING DEMO
The same in dollars, at a demo rate of 4 dollars per credit. Somatus's contract rate would apply in reality.

In [ ]:
%%sql -r by_client_usd
SELECT REGEXP_SUBSTR(QUERY_TAG, 'client=([A-Z_]+)', 1, 1, 'e') AS CLIENT,
       ROUND(SUM(CREDITS_ATTRIBUTED_COMPUTE), 4)        AS CREDITS,
       ROUND(SUM(CREDITS_ATTRIBUTED_COMPUTE) * 4.00, 2) AS ESTIMATED_USD_AT_4_PER_CREDIT
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATTRIBUTION_HISTORY
WHERE START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP()) AND QUERY_TAG ILIKE '%client=%'
GROUP BY 1 ORDER BY 2 DESC;

> **SAY THIS:** Now you can tell each health-plan contract what its data processing costs, and see which client's files are the expensive ones.

## Step 4 - Serverless costs are visible too

### RUN DURING DEMO
Credits by service type: warehouses, Snowpipe, dynamic tables, data quality checks and so on.

In [ ]:
%%sql -r serverless
SELECT SERVICE_TYPE, ROUND(SUM(CREDITS_USED), 3) AS CREDITS
FROM SNOWFLAKE.ACCOUNT_USAGE.METERING_HISTORY
WHERE START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY 1 ORDER BY 2 DESC;

### RUN DURING DEMO
Snowpipe cost per pipe and number of files loaded.

In [ ]:
%%sql -r pipe_cost
SELECT PIPE_NAME, ROUND(SUM(CREDITS_USED), 4) AS CREDITS, SUM(FILES_INSERTED) AS FILES
FROM SNOWFLAKE.ACCOUNT_USAGE.PIPE_USAGE_HISTORY
WHERE START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP()) GROUP BY 1;

## Step 5 - Guardrails: a budget with alerts

### RUN DURING DEMO
Show the monthly spending limit (200 credits).

In [ ]:
%%sql -r budget_limit
CALL SOMATUS_DEMO.OPS.SOMATUS_DE_BUDGET!GET_SPENDING_LIMIT();

### RUN DURING DEMO
Show which warehouses the budget covers.

In [ ]:
%%sql -r budget_resources
CALL SOMATUS_DEMO.OPS.SOMATUS_DE_BUDGET!GET_LINKED_RESOURCES();

### UI STEP - do this in the screen, not in the notebook
**Show the budget screen**

1. In the left menu, click **Admin**, then **Cost Management**.
2. Click the **Budgets** tab, then click **SOMATUS_DE_BUDGET**.
3. Show spend to date against the limit, the forecast line, and the notification settings.
4. Optionally click the **Consumption** tab and group by warehouse or tag to show the same breakdown visually.

> **SAY THIS:** Cost is visible by workload, by client and by feature, with a budget that warns you before you overspend.